# KI-Labor – Assignment 4: Transfer Learning

In [ ]:
import random
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, Subset, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from torchvision.models import EfficientNet_B5_Weights, ResNet18_Weights, efficientnet_b5, resnet18

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")  # Für Macs
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
torch.backends.cudnn.benchmark = True   # feste Bildgrößen -> cuDNN wählt einmal die schnellsten Kernel

print(f"PyTorch {torch.__version__} | Gerät: {DEVICE}")

## 1. Datensatz laden und ansehen

In [ ]:
# Food-101 einmalig herunterladen und entpacken (~5 GB, landet in ./data/food-101).
# Ist der Ordner schon vorhanden, passiert hier nichts.
DATA_ROOT = "./data"
for split in ("train", "test"):
    torchvision.datasets.Food101(root=DATA_ROOT, split=split, download=True)
print("Food-101 bereit")

In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 32
NUM_WORKERS = 8
VAL_FRACTION = 0.1


def food101_splits(train_tf, eval_tf):
    """Offizieller train-Split -> 90 % Training / 10 % Validierung (immer dieselben Indizes), test bleibt test."""
    train_full = torchvision.datasets.Food101(root=DATA_ROOT, split="train", transform=train_tf)
    val_full = torchvision.datasets.Food101(root=DATA_ROOT, split="train", transform=eval_tf)
    test = torchvision.datasets.Food101(root=DATA_ROOT, split="test", transform=eval_tf)
    perm = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(SEED)).tolist()
    n_val = int(VAL_FRACTION * len(perm))
    return Subset(train_full, perm[n_val:]), Subset(val_full, perm[:n_val]), test


def make_loader(dataset, batch_size=BATCH_SIZE, shuffle=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True)


eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
train_transform = transforms.Compose([transforms.RandomHorizontalFlip(), eval_transform])

trainset, valset, testset = food101_splits(train_transform, eval_transform)
trainloader = make_loader(trainset, shuffle=True)
valloader = make_loader(valset)
testloader = make_loader(testset)

classes = testset.classes
print(f"Train: {len(trainset)} | Val: {len(valset)} | Test: {len(testset)} Bilder")

In [ ]:
images, labels = next(iter(trainloader))
plt.figure(figsize=(12, 6))
plt.imshow(torchvision.utils.make_grid(images / 2 + 0.5).permute(1, 2, 0))   # Normalisierung rückgängig
plt.axis("off")
plt.show()
print(", ".join(classes[label] for label in labels))

In [ ]:
raw_train = torchvision.datasets.Food101(root=DATA_ROOT, split="train")
num_classes = len(raw_train.classes)

sizes, modes = [], Counter()
for file in raw_train._image_files:   # Image.open liest nur den Header, das geht schnell
    with Image.open(file) as img:
        sizes.append(img.size)
        modes[img.mode] += 1
widths, heights = zip(*sizes)
in_channels = 3   # alle Bilder sind RGB (siehe Modi)

counts = Counter(raw_train._labels[i] for i in trainset.indices)   # Klassenverteilung im Trainingsteil

print(f"Klassen: {num_classes}  (z.B. {raw_train.classes[:3]})")
print(f"Modi: {dict(modes)}")
print(f"Breite: {min(widths)}–{max(widths)} px, Höhe: {min(heights)}–{max(heights)} px, "
      f"{len(set(sizes))} verschiedene Größen")
print(f"Trainingsbilder pro Klasse: {min(counts.values())}–{max(counts.values())}")
print(f"Tensor-Shape fürs Custom CNN: {tuple(trainset[0][0].shape)}")

**Ergebnis der Analyse:** _wird nach dem Lauf ergänzt_

## 2. Custom CNN von Grund auf (Baseline)

Aufbau wie in Woche 3, aber mit `BatchNorm2d` nach jeder Faltung: 4× Conv-BN-ReLU, 2× MaxPool, danach 2 Fully-Connected-Schichten mit Dropout. Training mit SGD (lr 0.01, Momentum 0.9, Weight Decay 5e-4) über 10 Epochen.

In [ ]:
class BNNet(nn.Module):
    def __init__(self, in_channels, num_classes, img_size):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, 3, padding=1, bias=False)
        self.bn4 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128 * (img_size // 4) ** 2, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

In [ ]:
# bf16-Autocast für alle Modelle gleich (sonst passt EfficientNet_B5 bei 456 px nicht in den GPU-Speicher)
USE_AMP = DEVICE.type == "cuda"
criterion = nn.CrossEntropyLoss()
results = []   # eine Zeile pro Ansatz für die Vergleichstabelle


def autocast():
    return torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=USE_AMP)


def count_trainable(net):
    return sum(p.numel() for p in net.parameters() if p.requires_grad)


def evaluate(net, loader):
    net.eval()   # BatchNorm nutzt dann die gelernten Statistiken, Dropout ist aus
    loss_sum = correct = total = 0
    with torch.no_grad(), autocast():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = net(inputs).float()
            loss_sum += criterion(outputs, labels).item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return loss_sum / total, correct / total


def train(net, optimizer, train_loader, val_loader, epochs, run_name):
    """Gemeinsame Trainingsschleife für alle Ansätze. Loggt nach runs/food101/<run_name> (TensorBoard)
    und gibt die Lernkurven und die Trainingszeit zurück."""
    net.to(DEVICE)
    writer = SummaryWriter(f"runs/food101/{run_name}")
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.perf_counter()
    for epoch in range(epochs):
        net.train()
        loss_sum = correct = total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with autocast():
                outputs = net(inputs).float()
                loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        val_loss, val_acc = evaluate(net, val_loader)
        for key, value in zip(history, (loss_sum / total, correct / total, val_loss, val_acc)):
            history[key].append(value)
            writer.add_scalar(key, value, epoch)
        print(f"[{epoch + 1:2d}] train loss {loss_sum / total:.3f} acc {100 * correct / total:5.1f} % | "
              f"val loss {val_loss:.3f} acc {100 * val_acc:5.1f} %")
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    duration = time.perf_counter() - start
    writer.close()
    return history, duration


def report(model, strategy, resolution, epochs, trainable, train_time, test_acc, history):
    print(f"Test-Accuracy: {100 * test_acc:.2f} %  |  Trainingszeit: {train_time:.1f} s")
    results.append({"Modell": model, "Strategie": strategy, "Auflösung": f"{resolution} px", "Epochen": epochs,
                    "Trainierbare Parameter": trainable, "Trainingszeit [s]": round(train_time, 1),
                    "Test-Accuracy [%]": round(100 * test_acc, 2)})

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    epoch_axis = range(1, epochs + 1)
    for ax, metric in ((ax1, "loss"), (ax2, "acc")):
        ax.plot(epoch_axis, history[f"train_{metric}"], marker="o", label="train")
        ax.plot(epoch_axis, history[f"val_{metric}"], marker="o", label="val")
        ax.set(title=f"{model} ({strategy}) – {'Loss' if metric == 'loss' else 'Accuracy'}", xlabel="Epoche")
        ax.legend()
    plt.show()

In [ ]:
torch.manual_seed(SEED)
EPOCHS_CNN = 10
net = BNNet(in_channels, num_classes, IMG_SIZE)
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

history, train_time = train(net, optimizer, trainloader, valloader, EPOCHS_CNN, "bnnet")
_, test_acc = evaluate(net, testloader)
report("Custom CNN (BNNet)", "from scratch", IMG_SIZE, EPOCHS_CNN, count_trainable(net), train_time, test_acc, history)
del net, optimizer

## 3. Transfer Learning: ResNet18 und EfficientNet_B5

In [ ]:
ARCHS = {
    "ResNet18": (resnet18, ResNet18_Weights.DEFAULT),
    "EfficientNet_B5": (efficientnet_b5, EfficientNet_B5_Weights.DEFAULT),
}


def load_pretrained(arch):
    """Lädt das ImageNet-Modell und gibt (Backbone, neuer Kopf mit num_classes Ausgängen) zurück."""
    constructor, weights = ARCHS[arch]
    torch.manual_seed(SEED)
    net = constructor(weights=weights)
    if arch == "ResNet18":
        head = nn.Linear(net.fc.in_features, num_classes)
        net.fc = nn.Identity()
    else:
        net.classifier[1] = nn.Linear(net.classifier[1].in_features, num_classes)
        head = net.classifier   # Dropout + Linear
        net.classifier = nn.Identity()
    return net.to(DEVICE), head.to(DEVICE)


def extract_features(backbone, dataset, batch_size):
    backbone.eval()
    features, labels = [], []
    with torch.no_grad(), autocast():
        for inputs, targets in make_loader(dataset, batch_size):
            features.append(backbone(inputs.to(DEVICE)).float().cpu())
            labels.append(targets)
    return TensorDataset(torch.cat(features), torch.cat(labels))


def run_frozen(arch, epochs, batch_size=BATCH_SIZE):
    weights = ARCHS[arch][1]
    backbone, head = load_pretrained(arch)
    backbone.requires_grad_(False)
    train_set, val_set, test_set = food101_splits(weights.transforms(), weights.transforms())

    start = time.perf_counter()
    train_feats = extract_features(backbone, train_set, batch_size)
    val_feats = extract_features(backbone, val_set, batch_size)
    extract_time = time.perf_counter() - start
    print(f"Features für train+val berechnet in {extract_time:.1f} s, Feature-Dimension {train_feats[0][0].numel()}")

    optimizer = optim.Adam(head.parameters(), lr=1e-3)
    history, head_time = train(head, optimizer, DataLoader(train_feats, batch_size=256, shuffle=True),
                               DataLoader(val_feats, batch_size=256), epochs, f"{arch.lower()}_frozen")
    _, test_acc = evaluate(head, DataLoader(extract_features(backbone, test_set, batch_size), batch_size=256))
    report(arch, "fixed feature extractor", weights.transforms().crop_size[0], epochs,
           count_trainable(head), extract_time + head_time, test_acc, history)


def run_finetune(arch, epochs, batch_size=BATCH_SIZE):
    weights = ARCHS[arch][1]
    backbone, head = load_pretrained(arch)
    model = nn.Sequential(backbone, head)
    train_tf = transforms.Compose([transforms.RandomHorizontalFlip(), weights.transforms()])
    train_set, val_set, test_set = food101_splits(train_tf, weights.transforms())

    optimizer = optim.Adam([{"params": backbone.parameters(), "lr": 1e-4},
                            {"params": head.parameters(), "lr": 1e-3}])
    history, train_time = train(model, optimizer, make_loader(train_set, batch_size, shuffle=True),
                                make_loader(val_set, batch_size), epochs, f"{arch.lower()}_finetune")
    _, test_acc = evaluate(model, make_loader(test_set, batch_size))
    report(arch, "fine-tuning", weights.transforms().crop_size[0], epochs,
           count_trainable(model), train_time, test_acc, history)

    del model, optimizer
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

### 3.1 ResNet18 als Fixed Feature Extractor

In [ ]:
run_frozen("ResNet18", epochs=10)

### 3.2 ResNet18 Fine-Tuning

In [ ]:
run_finetune("ResNet18", epochs=2)

### 3.3 EfficientNet_B5 als Fixed Feature Extractor

In [ ]:
run_frozen("EfficientNet_B5", epochs=10, batch_size=16)

### 3.4 EfficientNet_B5 Fine-Tuning

In [ ]:
run_finetune("EfficientNet_B5", epochs=2, batch_size=16)

## 4. Vergleich aller Ansätze

Alle Werte stammen aus demselben Test-Split (25 250 Bilder), gemessen auf derselben GPU. Die Lernkurven aller Läufe lassen sich auch in TensorBoard vergleichen: `tensorboard --logdir runs/food101`.

In [ ]:
df = pd.DataFrame(results)
df["Zeit pro Epoche [s]"] = (df["Trainingszeit [s]"] / df["Epochen"]).round(1)
display(df.style.format({"Trainierbare Parameter": "{:,}", "Trainingszeit [s]": "{:.1f}",
                         "Test-Accuracy [%]": "{:.2f}", "Zeit pro Epoche [s]": "{:.1f}"}).hide(axis="index"))

labels = [f"{m}\n{s}" for m, s in zip(df["Modell"], df["Strategie"])]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col in zip(axes, ("Test-Accuracy [%]", "Trainingszeit [s]")):
    ax.barh(labels, df[col])
    ax.set(title=col)
    ax.invert_yaxis()
    for i, v in enumerate(df[col]):
        ax.text(v, i, f" {v:.1f}", va="center")
plt.tight_layout()
plt.show()

## 5. Interpretation

_wird nach dem Lauf ergänzt_